# GEF2017 全方法与层级一致性检查

本入口保留原文件名，分析范围为当前项目的 10 节点去重层级。
覆盖 FHF、HFF、HFFPS，1/3/6 步预测，以及全部已有协调方法。
逐节点精度、结构层级、语义评估层级、父子一致性和输入来源分别审计。

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from PPHF_modules.results_check.current_results import (
    prepare_current_results, core_accuracy_table, plot_training, plot_timing,
)

In [ ]:
DATASET = 'gef2017'
SOURCE_DIR = ROOT / 'fhf_dedup_gef2017'
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / DATASET
RECOMPUTE = False
tables = prepare_current_results(DATASET, root=ROOT, source_dir=SOURCE_DIR,
                                 output_dir=OUTPUT_DIR, force=RECOMPUTE)

## 节点与层级

结构层级用于父子一致性，语义评估层级用于精度汇总。
去重保留的叶节点可能在结构第 2 层，但在精度评估时归入底层。

In [ ]:
nodes = tables['per_node'][['unique_id','structural_level','level','role']].drop_duplicates()
display(nodes.sort_values(['structural_level','unique_id']))
display(tables['hierarchy_audit'])

## 全方法精度与逐节点结果

In [ ]:
display(tables['accuracy_by_level'])
display(tables['per_node'].head(40))
print('All node metrics:', OUTPUT_DIR / 'per_node.csv')

## 父子约束与归一化一致性

In [ ]:
display(tables['coherence_by_method'])
parents = tables['coherence_by_parent']
display(parents[['horizon','source_system','method_internal','parent_id','level',
                 'incoherence_total','mase_scale','MASI','n_time_points']])

## 数值阈值检查

以下将协调方法的 MASI 与指定阈值比较，逐项保留诊断结果。
精度优劣由上面的 MAE/MASE/RMSSE 表判断。

In [ ]:
COHERENCE_TOLERANCE = 1e-6
checks = tables['coherence_by_method'].loc[
    ~tables['coherence_by_method'].method_internal.isin(['base','naive'])].copy()
checks['within_tolerance'] = checks.MASI <= COHERENCE_TOLERANCE
display(checks[['horizon','source_system','method_internal','MASI','within_tolerance']])
checks.to_csv(OUTPUT_DIR / 'coherence_checks.csv', index=False)

## 输入来源与内容指纹

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'source_manifest.json').read_text())
display(pd.DataFrame(manifest['signature']['sources']))
display(pd.DataFrame([manifest['validation']]))
assert set(tables['hierarchy_audit'].nodes) == {10}
assert set(tables['hierarchy_audit'].horizon) == {1,3,6}
assert set(tables['hierarchy_audit'].source_system) == {'FHF','HFF','HFFPS'}